# Quickstart

Submit a GALFORM N-body run to SLURM in four steps.

**Requires:** `galform_execution` installed and access to COSMA with a compiled GALFORM source tree containing `build/galform2`, `*.input.ref`, and `replace_variable.csh`.

In [ ]:
from galform_execution.submit_galform_job import GalformSubmitter, _default_galform_dir

# Defaults to /cosma/apps/durham/$USER/galform — override if your tree is elsewhere
GALFORM_DIR = _default_galform_dir()

## Step 1 — Construct a submitter

Choose a simulation, model, snapshot (`iz`), and subvolume range (`nvol`). The submitter resolves cosmology, tree paths, and subvolume parameters from the bundled JSON configs.

In [ ]:
s = GalformSubmitter(
    GALFORM_DIR,
    nbody_sim="Mill2",
    model="lc16",
    iz=40,
    nvol="1-64",
    output_folder_name="MyProject",
)

print(f"Simulation : {s.nbody_sim}")
print(f"Model      : {s.model}")
print(f"Snapshots  : {s.iz_list}")
print(f"Subvolumes : {s.nvol_range}  ({s.nvol_count} subvolumes per snapshot)")
print(f"Output     : {s.models_dir}")
print(f"Logs       : {s.log_path}")

## Step 2 — Dry run

Print the generated `tcsh` script without touching the queue. Always do this before submitting.

In [ ]:
s.submit_all_jobs(dry_run=True)

## Step 3 — Inspect the scripts directly

Two scripts are involved:

- **`create_job_script(iz)`** — the bash wrapper that is submitted to `sbatch`. It requests `--cpus-per-task` (capped at the partition's CPUs per node) and forks one strided worker per CPU to cover all subvolumes in a single SLURM slot.
- **`_create_tcsh_script(iz)`** — the inner `tcsh` GALFORM script written to disk by `submit_job`. Contains the module loads, cosmological parameters, and pipeline stage invocations.

Inspect the bash wrapper to confirm SLURM directives and CPU allocation:

In [ ]:
script = s.create_job_script(iz=40)
print(script)

## Step 4 — Submit

`submit_all_jobs` loops over every snapshot in `iz_list` and returns SLURM job IDs.

In [ ]:
job_ids = s.submit_all_jobs(dry_run=False)
print(f"Submitted: {job_ids}")

---

**Next:** `02_simulations_models_and_pipeline.ipynb` — browsing configs, choosing simulations and models, customising pipeline stages and SLURM settings.